In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [4]:
hawaii = 'https://scholarspace.manoa.hawaii.edu/communities/533eb99e-2e59-47a5-9c4e-652352bfc5a6'

In [5]:
hawaii = requests.get(hawaii).text

In [6]:
hawaii_soup = BeautifulSoup(hawaii, 'html.parser')

In [7]:
def extract_hrefs(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    hrefs = [a['href'] for a in soup.find_all('a', class_='lead ng-star-inserted', href=True)]
    return hrefs

In [8]:
base_url = 'https://scholarspace.manoa.hawaii.edu/communities/533eb99e-2e59-47a5-9c4e-652352bfc5a6?cmcl.page='

# List to store all hrefs
all_hrefs = []

# Iterate over pages
for page_number in range(1, 10):  # Adjust the range as needed
    page_url = f'{base_url}{page_number}'
    hrefs = extract_hrefs(page_url)
    all_hrefs.extend(hrefs)

# Print all hrefs
for href in all_hrefs:
    print(href)

/collections/8e4c3ded-2f0b-44ad-98ff-6d93969f029e
/collections/bf4ed241-69d9-477a-81a3-1605d499316f
/collections/d49da7ee-c81f-4fed-acdb-efa9f0ed5970
/collections/7b05d7fa-597b-4a2c-918e-f68d29a91d86
/collections/4a1b41b0-8e05-48cd-b1cc-d198198b976c
/collections/1c44423e-902c-4e26-8c44-761cd017fa41
/collections/65d98780-f7ba-4523-8b37-70a37da197f2
/collections/686e028b-a6b0-44ab-a5a1-6757baad7ac5
/collections/4901cefe-4918-4ee9-8db6-06dff65280b0
/collections/3b819417-6911-4df2-990d-6bab4507448c
/collections/9433b5ba-f912-4275-aa10-e076b4f3f743
/collections/21c604e1-6719-4869-84e5-95cc6cb28c91
/collections/823bd02a-9436-402f-95f3-1f89315a30b0
/collections/82c0d02b-5c31-43e6-a581-66678f94b544
/collections/a1426c5f-3e3a-403d-893b-7b6ce96e5d1b
/collections/9d0c315c-ca16-4a59-94eb-b66c61e605e4
/collections/13d56c46-9bf4-4877-bdaf-bdcb44b32496
/collections/e1cf3b4d-ba28-4d2b-bad0-a49356b9060f
/collections/e23bb2a5-4764-4230-9681-3eafe639bf9e
/collections/df6553f8-1c51-42bc-b92c-d283d3f3e694


In [9]:
prefix = 'https://scholarspace.manoa.hawaii.edu'
for i in range(len(all_hrefs)):
    all_hrefs[i] = prefix + all_hrefs[i]

In [ ]:
def extract_hrefs(url, max_pages=30):
    all_hrefs = []
    visited_hrefs = set()
    page_num = 1
    while page_num <= max_pages:
        print(page_num)
        response = requests.get(url + f'?cp.page={page_num}')
        # Check if the response status code indicates success (200)
        if response.status_code != 200:
            break
        soup = BeautifulSoup(response.text, 'html.parser')
        links = soup.find_all('a', {'class': 'lead item-list-title dont-break-out ng-star-inserted'})
        inner_hrefs = [link['href'] for link in links]
        if not inner_hrefs:
            break
        for href in inner_hrefs:
            if href in visited_hrefs:
                # If we encounter a href that we've already visited, break the loop
                break
            all_hrefs.append(href)
            visited_hrefs.add(href)
        else:
            page_num += 1
            continue
        break  # Break the while loop if we encounter a visited href
    return all_hrefs


# Iterate over each webpage URL and extract href attributes
inner_hrefs = []
for url in chunk_5:
    hrefs = extract_hrefs(url)
    inner_hrefs.extend(hrefs)

In [17]:
for i in range(len(inner_hrefs)):
    inner_hrefs[i] = prefix + inner_hrefs[i] + '/full'

In [20]:
# Function to scrape content language and code from inner_href
def scrape_content_language(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    language_tag = soup.find('td', text='dc.content.language')
    language_code_tag = soup.find('td', text='dc.content.languagecode')
    if language_tag and language_code_tag:
        language = language_tag.find_next_sibling('td').text.strip()
        language_code = language_code_tag.find_next_sibling('td').text.strip()
        return language, language_code
    else:
        return None, None

# Function to scrape file name and size
def scrape_files_info(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    file_sections = soup.find_all('div', class_='file-section row mb-3 ng-star-inserted')
    file_names = []
    file_sizes = []
    for section in file_sections:
        name_tag = section.find('dt', text='Name:')
        size_tag = section.find('dt', text='Size:')
        if name_tag and size_tag:
            file_names.append(name_tag.find_next_sibling('dd').text.strip())
            file_sizes.append(size_tag.find_next_sibling('dd').text.strip())
    return file_names, file_sizes



In [ ]:
# Initialize lists to store scraped data
languages = []
language_codes = []
file_names_list = []
file_sizes_list = []

# Scrape data for each inner href
for index, href in enumerate(inner_hrefs):
    print(index)
    language, language_code = scrape_content_language(href)
    if language and language_code:
        languages.append(language)
        language_codes.append(language_code)
        file_names, file_sizes = scrape_files_info(href)
        file_names_list.append(file_names)
        file_sizes_list.append(file_sizes)
        origin_href = href

# Create DataFrame
data = {
    'Href': origin_href,
    'Content Language': languages,
    'Language Code': language_codes,
    'File Name': file_names_list ,
    'File Size': file_sizes_list 
}

results = pd.DataFrame(data)
results = results[results['File Size'] != '[]']


In [49]:
unit_multipliers = {
        'B': 1,
        'KB': 1024,
        'MB': 1024 ** 2,
        'GB': 1024 ** 3
    }

def convert_to_bytes(size_str):
    # Remove brackets and split into number and unit
    size_str = size_str.strip('[]')
    size_value, size_unit = size_str.split()
    size_value = float(size_value)
    
    # Convert to bytes
    bytes_size = int(size_value * unit_multipliers[size_unit])
    return bytes_size

def convert_list_to_total_bytes(size_list):
    total_bytes = sum(convert_to_bytes(size_str) for size_str in size_list)
    return total_bytes

bytes_to_gigabytes = lambda x: x / (1024 ** 3)

# Apply the conversion to the DataFrame column
results['Size_in_Bytes'] = results['File Size'].apply(convert_list_to_total_bytes)

In [84]:
def convert_to_bytes(size_str):
    # Conversion factors
    unit_multipliers = {
        'B': 1,
        'KB': 1024,
        'MB': 1024 ** 2,
        'GB': 1024 ** 3
    }
    
    # Remove brackets if present and split into number and unit
    size_str = size_str.strip('[]')
    size_parts = size_str.split()
    
    # Check if there are enough parts to unpack
    if len(size_parts) != 2:
        return None
    
    size_value, size_unit = size_parts
    
    try:
        size_value = float(size_value)
    except ValueError:
        return None
    
    # Convert to bytes
    bytes_size = int(size_value * unit_multipliers[size_unit])
    return bytes_size


def convert_list_to_total_bytes(size_list):
    total_bytes = sum(convert_to_bytes(size_str) for size_str in size_list)
    return total_bytes

In [85]:
results['Size_in_Bytes'] = results['File Size'].apply(convert_list_to_total_bytes)

In [88]:
results['gigabytes_column'] = results['Size_in_Bytes'].apply(bytes_to_gigabytes)

In [101]:
results =  results.groupby('Content Language')['gigabytes_column'].sum().reset_index()

In [102]:
results = results.sort_values(by='gigabytes_column', ascending=False)

In [106]:
results

,Unnamed: 0,Href,Content Language,Language Code,File Name,File Size,Size_in_Bytes,gigabytes_column
0,0,https://scholarspace.manoa.hawaii.edu/items/0e...,Sa’ani (possibly a dialect of Makasae),mkz,['TH1-461.WAV'],[492.67 MB],516601937,0.481123
1,1,https://scholarspace.manoa.hawaii.edu/items/0e...,Nauete,nxa,['TH1-460.WAV'],[404.08 MB],423708590,0.394609
2,2,https://scholarspace.manoa.hawaii.edu/items/0e...,Nauete,nxa,['TH1-459.WAV'],[266.24 MB],279172874,0.260000
3,3,https://scholarspace.manoa.hawaii.edu/items/0e...,Nauete,nxa,['TH1-458.WAV'],[594.22 MB],623084830,0.580293
4,4,https://scholarspace.manoa.hawaii.edu/items/0e...,Klon,kyo,['TH1-457.WAV'],[62.81 MB],65861058,0.061338
...,...,...,...,...,...,...,...,...
4519,1377,https://scholarspace.manoa.hawaii.edu/items/f6...,Dawoodi,dmk,['CD1-006.WAV'],[466.62 MB],489286533,0.455684
4520,1378,https://scholarspace.manoa.hawaii.edu/items/f6...,Dawoodi,dmk,"['CD1-005-A.WAV', 'CD1-005-B.WAV']","[78.51 MB, 67.24 MB]",152829951,0.142334
4521,1379,https://scholarspace.manoa.hawaii.edu/items/f6...,Dawoodi,dmk,['CD1-004.m4v'],[268.73 MB],281783828,0.262432
4522,1380,https://scholarspace.manoa.hawaii.edu/items/f6...,Dawoodi,dmk,['CD1-003.m4v'],[151.49 MB],158848778,0.147939


In [29]:
results = 

,Href,Content Language,Language Code,File Name,File Size,Size_in_Bytes
0,https://scholarspace.manoa.hawaii.edu/items/7d...,English,eng,[AMS1-009.pdf],[82.25 KB],84224
1,https://scholarspace.manoa.hawaii.edu/items/7d...,English,eng,[AMS1-008.pdf],[181.97 KB],186337
2,https://scholarspace.manoa.hawaii.edu/items/7d...,English,eng,[AMS1-007.pdf],[94.74 KB],97013
3,https://scholarspace.manoa.hawaii.edu/items/7d...,English,eng,[AMS1-005.pdf],[112.11 KB],114800
4,https://scholarspace.manoa.hawaii.edu/items/7d...,English,eng,[AMS1-006.pdf],[96.05 KB],98355
...,...,...,...,...,...,...
83,https://scholarspace.manoa.hawaii.edu/items/7d...,Pang,blr,"[XG1-005-01.wav, XG1-005-02.wav, XG1-005-03.wa...","[673.5 MB, 935.72 MB, 390.68 MB, 826.21 MB]",2963391117
84,https://scholarspace.manoa.hawaii.edu/items/7d...,Pang,blr,"[XG1-004-01.wav, XG1-004-02.wav]","[151.29 MB, 479.72 MB]",661661941
85,https://scholarspace.manoa.hawaii.edu/items/7d...,Hu,huo,"[XG1-003-01.wav, XG1-003-02.wav, XG1-003-03.wav]","[703.59 MB, 243.89 MB, 194.32 MB]",1197264075
86,https://scholarspace.manoa.hawaii.edu/items/7d...,Hu,huo,[XG1-002.wav],[586.32 MB],614801080


In [81]:
len(data['Content Language'])

10

In [84]:
data['Content Language']

["Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e"]

In [83]:
len(data['File Size'])

10